# Module 4 — Marx and the Machinery Question
## Capital, Labour, and the Distribution of Income

**From Smith to Simulation: Computing the Ideas that Built Economics**  
The University of Edinburgh · School of Economics

---

## Part I — The History

### The Dark Side of the Pin Factory

Adam Smith marvelled at the pin factory's productivity. But by the 1840s, the Industrial Revolution had raised an uncomfortable question: **who benefits?**

Factories were producing more than ever before, but the workers inside them laboured 14-hour days in dangerous conditions for wages that barely covered food and rent. Children as young as five worked in coal mines. Meanwhile, factory owners grew spectacularly rich. The gap between capital and labour had never been wider.

This was the world that **Karl Marx** (1818–1883) set out to explain.

### Marx's Analysis

Marx, writing from the reading room of the British Museum in London, produced *Capital* (1867) — a massive, detailed analysis of how capitalism works and, he argued, why it would eventually destroy itself.

His key ideas for our purposes:

**1. The Labour Theory of Value.** Marx (following Ricardo) argued that the value of a good comes from the labour required to produce it. A chair is worth more than a stick because more labour went into making it.

**2. Surplus Value.** Workers produce more value than they receive in wages. The difference — **surplus value** — is captured by the capitalist as profit. A factory worker might produce £10 worth of goods per hour but earn only £3. The remaining £7 is surplus value.

**3. The Machinery Question.** As capitalists invest in machines to increase productivity, they replace workers with capital. Marx called this the "organic composition of capital" rising. The consequence: workers' share of income falls, unemployment rises, and wages are pushed down.

> *"The essential difference between the various economic forms of society, between, for instance, a society based on slave-labour, and one based on wage-labour, lies only in the mode in which this surplus labour is in each case extracted from the actual producer, the labourer."*  
> — Karl Marx, *Capital*, Volume I, Chapter 9

### The Machinery Question in the 21st Century

Marx's "machinery question" — does technology help or hurt workers? — is more relevant today than ever. Since the 1980s, the **labour share of income** (the fraction of GDP going to wages rather than profits) has been declining across developed economies. Automation, globalisation, and now artificial intelligence raise the same concerns Marx identified in 1867.

We can't resolve this debate in one module, but we can build a model that formalises the question and lets us explore the forces at work.

### Edinburgh Connection

Edinburgh's own Industrial Revolution — the expansion of printing, brewing (the city was once Britain's second-largest brewing centre), and engineering — was the backdrop for these debates. The Edinburgh Review, founded in 1802, was one of the key journals where political economists debated the effects of machinery on employment.

---

## Part II — The Computation

### Setting Up

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### The Production Function with Capital and Labour

We model output using a **Cobb-Douglas production function**:

$$Y = A \cdot K^\alpha \cdot L^{1-\alpha}$$

where $K$ is capital (machines, factories), $L$ is labour (workers), $A$ is total factor productivity, and $\alpha \in (0,1)$ is the **capital share** — the fraction of income going to capital owners.

In competitive markets, factors are paid their marginal products:
- **Wage:** $w = (1-\alpha) \cdot A \cdot K^\alpha \cdot L^{-\alpha} = (1-\alpha) \cdot Y/L$
- **Rental rate of capital:** $r = \alpha \cdot A \cdot K^{\alpha-1} \cdot L^{1-\alpha} = \alpha \cdot Y/K$

The **labour share** is $wL/Y = 1 - \alpha$ and the **capital share** is $rK/Y = \alpha$. Under Cobb-Douglas, these are constant — a result that was roughly true from 1950 to 1980 but has since broken down.

In [ ]:
# Parameters
A = 1.0
alpha = 0.33   # capital share
L = 100        # fixed labour supply

def output(K, L=L, A=A, alpha=alpha):
    return A * K**alpha * L**(1 - alpha)

def wage(K, L=L, A=A, alpha=alpha):
    return (1 - alpha) * A * K**alpha * L**(-alpha)

def rental_rate(K, L=L, A=A, alpha=alpha):
    return alpha * A * K**(alpha - 1) * L**(1 - alpha)

# How wages and profits change as capital accumulates
K_range = np.linspace(10, 500, 200)

Y = output(K_range)
total_wages = wage(K_range) * L
total_profits = rental_rate(K_range) * K_range

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(K_range, Y, color='seagreen', linewidth=2)
axes[0].set_xlabel('Capital (K)', fontsize=11)
axes[0].set_ylabel('Output (Y)', fontsize=11)
axes[0].set_title('Total Output', fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, wage(K_range), color='steelblue', linewidth=2, label='Wage per worker')
axes[1].plot(K_range, rental_rate(K_range), color='coral', linewidth=2, label='Return on capital')
axes[1].set_xlabel('Capital (K)', fontsize=11)
axes[1].set_ylabel('Factor price', fontsize=11)
axes[1].set_title('Wages vs Return on Capital', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

axes[2].plot(K_range, total_wages, color='steelblue', linewidth=2, label='Total wages (wL)')
axes[2].plot(K_range, total_profits, color='coral', linewidth=2, label='Total profits (rK)')
axes[2].set_xlabel('Capital (K)', fontsize=11)
axes[2].set_ylabel('Income', fontsize=11)
axes[2].set_title('Income Distribution', fontsize=12)
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("As capital accumulates:")
print("  - Output rises (but with diminishing returns)")
print("  - Wages RISE (more capital per worker makes each worker more productive)")
print("  - Return on capital FALLS (diminishing returns to capital)")
print(f"  - Labour share stays constant at {1-alpha:.0%} (a Cobb-Douglas property)")

### Capital Accumulation Over Time

Marx was interested in the *dynamics* of capitalism — how the system evolves. Capitalists reinvest their profits, accumulating more and more capital. Let's simulate this:

$$K_{t+1} = (1 - \delta) K_t + s \cdot r_t \cdot K_t$$

where $\delta$ is depreciation (machines wear out) and $s$ is the fraction of profits that capitalists reinvest.

In [ ]:
# Capital accumulation simulation
T = 200
delta = 0.05    # depreciation rate (5% per year)
s = 0.80        # capitalists reinvest 80% of profits
K0 = 50         # initial capital

def simulate_accumulation(K0, L, A, alpha, delta, s, T):
    K = np.empty(T)
    K[0] = K0
    Y_t = np.empty(T)
    w_t = np.empty(T)
    r_t = np.empty(T)
    labour_share = np.empty(T)
    
    for t in range(T):
        Y_t[t] = output(K[t], L, A, alpha)
        w_t[t] = wage(K[t], L, A, alpha)
        r_t[t] = rental_rate(K[t], L, A, alpha)
        labour_share[t] = w_t[t] * L / Y_t[t]
        
        if t < T - 1:
            profits = r_t[t] * K[t]
            investment = s * profits
            K[t+1] = (1 - delta) * K[t] + investment
    
    return K, Y_t, w_t, r_t, labour_share

K_sim, Y_sim, w_sim, r_sim, ls_sim = simulate_accumulation(K0, L, A, alpha, delta, s, T)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0,0].plot(K_sim, color='seagreen', linewidth=1.5)
axes[0,0].set_title('Capital Stock', fontsize=12)
axes[0,0].set_ylabel('K')
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(Y_sim, color='steelblue', linewidth=1.5)
axes[0,1].set_title('Output', fontsize=12)
axes[0,1].set_ylabel('Y')
axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(w_sim, color='steelblue', linewidth=1.5, label='Wage')
axes[1,0].set_title('Wage per Worker', fontsize=12)
axes[1,0].set_ylabel('w')
axes[1,0].set_xlabel('Year')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(r_sim, color='coral', linewidth=1.5)
axes[1,1].set_title('Rate of Profit (Return on Capital)', fontsize=12)
axes[1,1].set_ylabel('r')
axes[1,1].set_xlabel('Year')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Capital Accumulation Under Cobb-Douglas', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Initial wage: {w_sim[0]:.3f}, Final wage: {w_sim[-1]:.3f}")
print(f"Initial rate of profit: {r_sim[0]:.3f}, Final: {r_sim[-1]:.3f}")
print(f"\nMarx's \"tendency of the rate of profit to fall\" is visible here:")
print(f"as capital accumulates, the return on capital declines (diminishing returns).")
print(f"But wages RISE — which Marx's model didn't fully predict.")

### Beyond Cobb-Douglas: When the Labour Share Falls

The Cobb-Douglas function gives a constant labour share — which doesn't match the recent data. Since 1980, labour's share of income has fallen in most developed countries. To capture this, we need a production function where capital and labour are **not perfectly substitutable**.

The **CES (Constant Elasticity of Substitution)** production function does this:

$$Y = A \left[ \alpha K^{\frac{\sigma-1}{\sigma}} + (1-\alpha) L^{\frac{\sigma-1}{\sigma}} \right]^{\frac{\sigma}{\sigma-1}}$$

The parameter $\sigma$ (sigma) is the **elasticity of substitution** — how easily capital can replace labour.

- $\sigma = 1$: Cobb-Douglas (constant labour share)
- $\sigma > 1$: Capital and labour are easy to substitute. As capital accumulates, the labour share **falls**.
- $\sigma < 1$: Capital and labour are complements. As capital accumulates, the labour share **rises**.

In [ ]:
def ces_output(K, L, A, alpha, sigma):
    """CES production function."""
    rho = (sigma - 1) / sigma
    return A * (alpha * K**rho + (1 - alpha) * L**rho) ** (1 / rho)

def ces_labour_share(K, L, A, alpha, sigma):
    """Labour share under CES production."""
    rho = (sigma - 1) / sigma
    Y = ces_output(K, L, A, alpha, sigma)
    # MPL = dY/dL = A * (1-alpha) * L^(rho-1) * [...] ^ (1/rho - 1)
    inner = alpha * K**rho + (1 - alpha) * L**rho
    MPL = A * (1 - alpha) * L**(rho - 1) * inner**(1/rho - 1)
    return MPL * L / Y

# Compare labour share for different sigma as K grows
K_range = np.linspace(50, 1000, 200)

fig, ax = plt.subplots(figsize=(9, 5))

for sigma, color, ls in [(0.5, 'steelblue', 'Complements (σ=0.5)'),
                          (1.01, 'black', 'Cobb-Douglas (σ≈1)'),
                          (1.5, 'coral', 'Substitutes (σ=1.5)'),
                          (2.0, 'red', 'Easy substitution (σ=2)')]:
    shares = [ces_labour_share(K, L, A, 0.33, sigma) for K in K_range]
    ax.plot(K_range, shares, color=color, linewidth=2, label=ls)

ax.set_xlabel('Capital stock (K)', fontsize=12)
ax.set_ylabel('Labour share of income', fontsize=12)
ax.set_title('The Machinery Question: Does Capital Accumulation Hurt Workers?', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("When σ > 1 (capital and labour are substitutes), accumulating more capital")
print("REDUCES the labour share — exactly what Marx predicted and what we see post-1980.")
print("\nRecent estimates put σ between 1.2 and 1.6 for the US economy.")

### Historical Labour Share Data

Let's see how the real-world labour share has evolved. We'll use stylised data based on published estimates.

In [ ]:
# Stylised US labour share data (based on BLS and Karabarbounis & Neiman 2014)
years = np.array([1950, 1955, 1960, 1965, 1970, 1975, 1980,
                   1985, 1990, 1995, 2000, 2005, 2010, 2015, 2020])
labour_share_us = np.array([0.65, 0.65, 0.66, 0.66, 0.66, 0.65, 0.65,
                             0.64, 0.64, 0.63, 0.63, 0.61, 0.59, 0.58, 0.57])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(years, labour_share_us * 100, 'o-', color='steelblue', linewidth=2, markersize=6)
ax.fill_between(years, labour_share_us * 100, 55, alpha=0.1, color='steelblue')

ax.axvline(1980, color='red', linestyle='--', alpha=0.4, label='c. 1980: trend break')
ax.axhline(66, color='gray', linestyle=':', alpha=0.5, label='"Kaldor fact" (≈ 66%)')

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Labour share (%)', fontsize=12)
ax.set_title('US Labour Share of Income, 1950–2020', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(55, 70)
plt.tight_layout()
plt.show()

print("For decades, the labour share was roughly constant at ~66% — a 'Kaldor fact'.")
print("Since ~1980, it has declined by nearly 10 percentage points.")
print("This trend is visible in the US, Europe, Japan, and even China.")
print("\nMarx might say: 'I told you so.'")

---

## Part III — Exercises

### Exercise 1 — Marx's Falling Rate of Profit

Marx predicted that the rate of profit would tend to fall as capitalism matured. Let's test this using our model.

**(a)** Simulate capital accumulation for 300 periods using Cobb-Douglas ($\alpha=0.33$, $L=100$, $A=1$, $\delta=0.05$, $s=0.8$, $K_0=50$). Plot the rate of profit $r_t$ over time. Does it fall?

**(b)** Compute the steady-state capital stock (where investment equals depreciation: $s \cdot r \cdot K = \delta \cdot K$). Show that $r_{\text{ss}} = \delta / s$.

**(c)** Now add technological progress: $A_t = A_0 \cdot e^{0.01 t}$. Simulate again. Does the rate of profit still fall to the same level? Marx ignored sustained technological progress — does it save capitalism from the falling profit rate?

**(d)** In 2–3 sentences, assess Marx's prediction in light of your simulations.

In [ ]:
# Your answer here


### Exercise 2 — Automation and the CES Function

Suppose the elasticity of substitution increases from $\sigma = 1$ (Cobb-Douglas) to $\sigma = 1.5$ due to automation technologies (robots can now do many jobs previously done by humans).

**(a)** With $K = 200$, $L = 100$, $A = 1$, $\alpha = 0.33$, compute the labour share under $\sigma = 0.8, 1.0, 1.2, 1.5, 2.0$. Present as a table.

**(b)** For $\sigma = 1.5$, simulate capital accumulation over 200 periods (start at $K_0 = 50$, use $s = 0.3$, $\delta = 0.05$). Plot the labour share over time.

**(c)** At what level does the labour share stabilise? How does this compare to the Cobb-Douglas case?

In [ ]:
# Your answer here


### Exercise 3 — Was Marx Right About Wages?

Marx predicted that wages would be pushed to subsistence as capital accumulated. The evidence is more nuanced: real wages have risen enormously since 1850, but wages as a *share* of income have fallen recently.

**(a)** Using the Cobb-Douglas simulation from Part II, plot *real wages* $w_t$ alongside the *labour share* $w_t L / Y_t$ over time. Can wages rise while the labour share stays constant?

**(b)** Now add population growth: $L_t = L_0 \cdot e^{0.005t}$. Simulate capital accumulation and wages over 200 periods. Do wages still rise? How does population growth affect the result?

**(c)** In 2–3 sentences, explain where Marx was right and where he was wrong about wages.

In [ ]:
# Your answer here


---

## Part IV — Quiz

### Conceptual Questions

**Q1.** Marx's "surplus value" refers to:

(a) The price markup that firms charge over cost  
(b) The difference between the value workers produce and the wages they receive  
(c) Excess inventory that goes unsold  
(d) Government tax revenue

**Q2.** The "machinery question" in classical economics asks:

(a) Whether steam engines are more efficient than waterwheels  
(b) Whether technological change benefits or harms workers  
(c) How to optimally maintain factory equipment  
(d) Whether agriculture or manufacturing drives growth

**Q3.** Marx predicted that the rate of profit would tend to fall because:

(a) Competition would drive prices down  
(b) As more capital is accumulated, diminishing returns reduce the return on each unit  
(c) Workers would demand higher wages  
(d) Government regulation would squeeze profits

**Q4.** The labour share of income is defined as:

(a) The number of workers divided by total population  
(b) Total wages divided by GDP  
(c) Average wage divided by average productivity  
(d) Employment rate times hours worked

**Q5.** Since 1980, the labour share of income in developed countries has:

(a) Remained constant (a "Kaldor fact")  
(b) Risen steadily  
(c) Declined by roughly 8–10 percentage points  
(d) Fluctuated without trend

### Computational Questions

**Q6.** Under Cobb-Douglas production $Y = AK^{0.33}L^{0.67}$, the labour share equals:

(a) 0.33  
(b) 0.67  
(c) It depends on K and L  
(d) 0.50

**Q7.** In the CES production function, if $\sigma > 1$ and capital increases, the labour share:

(a) Rises  
(b) Falls  
(c) Stays constant  
(d) First rises then falls

**Q8.** The marginal product of labour under Cobb-Douglas is $(1-\alpha) Y/L$. If capital doubles while labour stays fixed, wages:

(a) Stay the same  
(b) Rise by a factor of $2^\alpha$  
(c) Double  
(d) Fall

**Q9.** The elasticity of substitution $\sigma$ measures:

(a) How fast capital depreciates  
(b) How easily firms can replace labour with capital  
(c) The rate of technological progress  
(d) The speed of wage adjustment

**Q10.** Marx argued workers would be immiserated under capitalism. Under Cobb-Douglas with capital accumulation, real wages actually:

(a) Fall, confirming Marx  
(b) Rise, as more capital makes each worker more productive  
(c) Stay constant at subsistence  
(d) Depend entirely on union bargaining power

### Quiz Answers

<details>
<summary>Click to reveal answers</summary>

**Q1.** (b) Surplus value is the gap between the value workers create through their labour and what they receive as wages — the source of profit in Marx's framework.

**Q2.** (b) Whether the introduction of machinery (and technology more broadly) helps or hurts the working class was a central debate among classical economists.

**Q3.** (b) As capitalists invest more and more, diminishing returns to capital reduce the profit rate. Marx called this the "tendency of the rate of profit to fall."

**Q4.** (b) Labour share = total compensation of workers / GDP. It measures what fraction of national income goes to labour versus capital.

**Q5.** (c) The labour share has declined significantly in most developed economies since around 1980, contradicting the long-standing "Kaldor fact" that it was roughly constant.

**Q6.** (b) Under Cobb-Douglas, the labour share is always $1-\alpha = 0.67$, regardless of $K$ and $L$.

**Q7.** (b) When $\sigma > 1$, capital and labour are substitutes. As $K$ rises, firms shift toward capital-intensive production and the labour share falls.

**Q8.** (b) $w = (1-\alpha)AK^\alpha L^{-\alpha}$. Doubling $K$ multiplies wages by $2^\alpha = 2^{0.33} \approx 1.26$ — a 26% increase.

**Q9.** (b) $\sigma$ measures how easily firms can substitute capital for labour (and vice versa). Higher $\sigma$ means easier substitution.

**Q10.** (b) Under Cobb-Douglas, more capital raises the marginal product of labour, so real wages rise even as the labour share stays constant. Marx was wrong about the level of wages but may have been right about the share.

</details>

---

## Further Reading

- Marx, K. *Capital*, Volume I (1867), Chapters 1, 7, 15 (selections).
- Heilbroner, R. *The Worldly Philosophers*, Chapter 6 ("The Inexorable System of Karl Marx").
- Piketty, T. *Capital in the Twenty-First Century* (2014), Chapter 6 (on the capital-labour split).
- Karabarbounis, L. & Neiman, B. "The Global Decline of the Labor Share" (QJE, 2014).

---

*Next week: The Marginalist Revolution — how economics shifted from class-based analysis to individual optimisation in the 1870s.*